# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Print key metadata fields, using the metadata object attributes
print(f"Name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'N/A')}\n")
print(f"Cite As: {getattr(dataset.metadata, 'citeAs', 'N/A')}\n")
print(f"Keywords: {getattr(dataset.metadata, 'keywords', 'N/A')}")
print(f"License: {getattr(dataset.metadata, 'license', 'N/A')}")

## 2. Data Overview
Explore available record sets and their fields using their `@id` fields.

**Note:** All references are made via the official `@id` for traceability.

In [ ]:
# List record sets and their fields
print('\nRecord Sets:')
record_sets = dataset.metadata.recordSet
if not record_sets or len(record_sets) == 0:
    # Dataset has no declared record sets (may be older Croissant or implicit CSV/TSV structure)
    print('No record sets found in top-level metadata. Attempting to infer from DataFiles...')
    # Try extracting record sets from the data files (for generic tabular data)
    if hasattr(dataset.metadata, 'distribution') and len(dataset.metadata.distribution) > 0:
        print('Files available in distribution:')
        for fileobj in dataset.metadata.distribution:
            print(f'- @id: {getattr(fileobj, "@id", None)}, Name: {getattr(fileobj, "name", "")}, URL: {getattr(fileobj, "contentUrl", "")}, Encoding: {getattr(fileobj, "encodingFormat", "")}' )
else:
    for rs in record_sets:
        print(f'- RecordSet @id: {rs["@id"]}, name: {getattr(rs, "name", "") }')
        # List fields for this record set
        if hasattr(rs, 'field'):
            print(f'    Fields:')
            for fld in rs.field:
                print(f'      - Field @id: {fld["@id"]}, name: {fld.get("name", "")}, type: {fld.get("dataType", "")}')

## 3. Data Extraction
Load data from a specific record set or table in the dataset. If the dataset does not have an explicit `recordSet`, we use the first available data file (from `distribution`).

All `@id` references below are taken from the overview above.

In [ ]:
# --- Find a data table or record set for extraction ---
record_set_id = None
df_key = None
# Try to pick a record set @id if available, otherwise pick the first data file
if hasattr(dataset.metadata, 'recordSet') and dataset.metadata.recordSet:
    record_sets = dataset.metadata.recordSet
    record_set_id = record_sets[0]["@id"]
    df_key = record_set_id
    print(f"Selected record set: {record_set_id}")
else:
    # Fallback: use the first distribution object
    distributions = getattr(dataset.metadata, 'distribution', [])
    if distributions:
        record_set_id = distributions[0]["@id"]
        df_key = record_set_id
        print(f"Selected DataFile: {record_set_id}")
    else:
        raise Exception("No record sets or data files found in the Croissant schema.")

# ---- Extract records ----
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)

# Print column list and a sample
print(f"\nColumns in data table (@id={record_set_id}):\n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, outlier removal, and grouping.

In [ ]:
# Let's inspect numeric fields in the DataFrame
numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric columns candidates: {numeric_candidates}")

# For demonstration, try using 'MW kDa', 'Normalized_Abund._Control_1', etc. as possible numeric fields
selected_numeric_field = None
for col in ['MW kDa', 'Normalized_Abund._Control_1', 'Coverage_%', 'PeptideCount']:
    if col in df.columns:
        selected_numeric_field = col
        break
if not selected_numeric_field and numeric_candidates:
    selected_numeric_field = numeric_candidates[0]

if selected_numeric_field is None:
    raise Exception("No numeric field found for EDA.")

print(f"Selected numeric field for analysis: {selected_numeric_field}")

group_field = None
for grp in ['Gene Symbol', 'Organism', 'Description', 'Accession']:
    if grp in df.columns:
        group_field = grp
        break

# --- Filter records with the numeric field greater than threshold ---
threshold = df[selected_numeric_field].mean() if pd.api.types.is_numeric_dtype(df[selected_numeric_field]) else 10
filtered_df = df[df[selected_numeric_field] > threshold]
print(f"Filtered records with {selected_numeric_field} > {threshold}:")
print(filtered_df.head())

# --- Normalize the field ---
filtered_df = filtered_df.copy()  # silence SettingWithCopyWarning
filtered_df[f"{selected_numeric_field}_normalized"] = (
    filtered_df[selected_numeric_field] - filtered_df[selected_numeric_field].mean()
) / filtered_df[selected_numeric_field].std()

print(f"\nNormalized '{selected_numeric_field}' for filtered records:")
print(filtered_df[[selected_numeric_field, f"{selected_numeric_field}_normalized" ]].head())

# --- Group by a key attribute ---
if group_field:
    grouped_df = filtered_df.groupby(group_field)[selected_numeric_field].mean().reset_index()
    print(f"\nGrouped data by {group_field} (mean {selected_numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and relationship to groupings (if possible).

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
df[selected_numeric_field].hist(bins=30, alpha=0.7)
plt.xlabel(selected_numeric_field)
plt.ylabel('Count')
plt.title(f'Distribution of {selected_numeric_field}')
plt.grid(True)
plt.show()

if group_field and group_field in filtered_df.columns:
    top_group = filtered_df[group_field].value_counts().index[:5]
    plot_df = filtered_df[filtered_df[group_field].isin(top_group)]
    plt.figure(figsize=(10, 5))
    plot_df.boxplot(column=selected_numeric_field, by=group_field)
    plt.title(f'{selected_numeric_field} by {group_field} (top 5)')
    plt.suptitle("")
    plt.ylabel(selected_numeric_field)
    plt.xlabel(group_field)
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset using its public Croissant schema, listed available record sets and data files via their `@id` fields, extracted tabular data, and performed basic EDA.

- Key numeric field analyzed: the field with the most available data (e.g., molecular weight or abundance values).
- The data shows variation in measured values, and can be grouped by gene symbol or accession.
- Further statistical and domain-specific analyses are enabled by the standardized schema and programmatic data access with `mlcroissant`.

Refer to the Croissant documentation for advanced data connections, provenance, or semantic queries, all by `@id` reference.